In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np
from scipy.stats import norm  # 用于高斯函数D(T)计算
import matplotlib.pyplot as plt

# 设备配置（CPU/GPU）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)  # 固定随机种子
np.random.seed(42)

#### 几何参数

In [2]:
d0 = 0.05  # 内管直径（m）
d1 = 0.13  # 外壳直径（m）
r0 = d0 / 2  # 内管半径
r1 = d1 / 2  # 外壳半径

#### 材料热物性参数

In [3]:
# PCM（石蜡）
rho_s = 880.0    # 固相密度 (kg/m³)
rho_l = 760.0    # 液相密度 (kg/m³)
cp_s = 2180.0    # 固相定压比热容 (J/(kg·K))
cp_l = 2390.0    # 液相定压比热容 (J/(kg·K))
lambda_s = 0.4   # 固相导热系数 (W/(m·K))
lambda_l = 0.15  # 液相导热系数 (W/(m·K))
mu_l = 0.001     # 液相粘度 (kg/(m·s))
L = 255000.0     # 相变潜热 (J/kg)，文档L=255kJ/kg转换为J/kg
Tpc = 316.15     # 相变温度 (K)
DeltaT = 6.0     # 相变温度区间 (K)

# 高导热材料（铜）
rho_Cu = 8960.0  # 密度 (kg/m³)
lambda_Cu = 400.0  # 导热系数 (W/(m·K))
cp_Cu = 385.0    # 定压比热容 (J/(kg·K))
mu_Cu = 1e10     # 铜为固体，粘度取极大值抑制流动

# 外壳材料（铝）- 仅用于边界，此处拓扑优化设计域为PCM+铜，铝不参与设计
cp_Al = 879.0    # 定压比热容 (J/(kg·K))

#### 物理模型参数

In [4]:
Am = 1e5         # 糊状区常数（文档未明确）
epsilon = 0.001  # 避免分母为0
xi = 1.0         # 粘度函数常数
alpha = 1e-4     # PCM体胀系数 (1/K)，自然对流计算需要
g = 9.81         # 重力加速度 (m/s²)

#### 拓扑优化参数

In [5]:
phi_total = 0.3  # 高导热材料体积比约束（预设，工程常用0.3）
case = 3         # 优化目标选择：1=平均温度，2=温度均方差，3=多目标
w1, w2, w3 = 1.0, 1.0, 1.0  # 多目标权重

#### 训练超参数 

In [6]:
N_mass = 10000   # 质量守恒方程采样点数量
N_mom = 10000    # 动量方程采样点数量
N_heat = 10000   # 传热方程采样点数量
N_IC = 8000      # 初始条件采样点数量
N_BC1 = 3000     # 内管壁边界采样点数量
N_BC2 = 3000     # 外壳边界采样点数量
N_rho = 10000    # 拓扑设计变量采样点数量
N_vol = 5000     # 体积比约束采样点数量
N_obj = 10000    # 优化目标采样点数量

lambda1 = 1.0    # PDE损失权重
lambda2 = 1.0    # IC/BC损失权重
lambda3 = 100.0  # 拓扑约束损失权重
lambda4 = 0.1    # 优化目标损失权重

In [7]:
class ResidualBlock(nn.Module):
    """残差块：缓解深层网络梯度消失，提升表达能力"""
    def __init__(self, dim):
        super(ResidualBlock, self).__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.tanh = nn.Tanh()  # 保持与原网络一致的激活函数
    
    def forward(self, x):
        residual = x  # 残差连接：保留输入特征
        out = self.tanh(self.fc1(x))
        out = self.fc2(out)
        return self.tanh(out + residual)  # 残差相加后激活

In [8]:
class TopoPINN(nn.Module):
    def __init__(self, hidden_layers=6, hidden_dim=256):
        super(TopoPINN, self).__init__()
        # 主网络：处理随时间变化的变量 (ux, uy, p, T)
        self.main_net = nn.Sequential(
            nn.Linear(3, hidden_dim),  # 输入(x,y,τ)
            nn.Tanh(),
            *[ResidualBlock(hidden_dim) for _ in range(hidden_layers)],
            nn.Linear(hidden_dim, 4)  # 输出ux, uy, p, T
        )
        
        # 拓扑网络：仅处理空间变量，输出不随时间变化
        self.topo_net = nn.Sequential(
            nn.Linear(2, hidden_dim),  # 仅输入(x,y)
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),  # 输出ρ_x
            nn.Sigmoid()
        )
    
    def forward(self, x):
        # x: [batch_size, 3]，对应(x,y,τ)
        # 提取空间坐标
        x_space = x[:, 0:2]
        
        # 主网络输出
        main_out = self.main_net(x)
        ux = main_out[:, 0:1] * 1e-3  # 速度量级：1e-3 m/s
        uy = main_out[:, 1:2] * 1e-3
        p = main_out[:, 2:3] * 1e3    # 压力量级：1e3 Pa
        T = main_out[:, 3:4] * 50 + 300  # 温度量级：250-350 K
        
        # 拓扑网络输出（仅依赖于空间坐标）
        rho_x = self.topo_net(x_space)
        
        return ux, uy, p, T, rho_x
    
    def compute_phi_from_T(self, T):
        """根据温度计算液相率φ(T)"""
        # 相变温度Tpc = 316.15K，相变区间ΔT = 6K
        T_lower = Tpc - DeltaT/2  # 313.15K
        T_upper = Tpc + DeltaT/2  # 319.15K
        
        # 线性相变模型
        phi = (T - T_lower) / (T_upper - T_lower)
        phi = torch.clamp(phi, 0.0, 1.0)
        return phi

In [9]:
def compute_thermo_props(T, rho_x):
    """
    计算混合材料的热物性参数（PCM+铜）
    输入：T（温度）、rho_x（拓扑设计变量）
    输出：rho_total, lambda_total, mu_total, cp_total, a（热扩散率）
    """
    # -------------------------- 1. 液相率φ计算--------------------------
    phi = torch.clamp((T - (Tpc - DeltaT/2)) / DeltaT, 0.0, 1.0)  # φ∈[0,1]

    # -------------------------- 2. 密度ρ_total--------------------------
    # PCM密度：ρ_PCM = ρ_s + (ρ_l - ρ_s)*φ
    rho_PCM = rho_s + (rho_l - rho_s) * phi
    # 混合密度：ρ_total = ρx*ρ_Cu + (1-ρx)*ρ_PCM
    rho_total = rho_x * rho_Cu + (1.0 - rho_x) * rho_PCM

    # -------------------------- 3. 导热系数λ_total--------------------------
    lambda_PCM = lambda_s + (lambda_l - lambda_s) * phi
    # 混合导热系数：调和平均（更符合物理实际）
    lambda_reciprocal = rho_x/lambda_Cu + (1.0 - rho_x)/lambda_PCM
    lambda_total = 1.0 / (lambda_reciprocal + 1e-10)

    # -------------------------- 4. 粘度mu_total--------------------------
    # 糊状区源项S_t(T)
    S_t = Am * ((1.0 - phi) ** 2) / (phi ** 2 + epsilon)
    # PCM粘度
    mu_PCM = mu_l + S_t * 1.0  # ξ = 1.0 m²
    mu_PCM = torch.clamp(mu_PCM, 1e-6, 1e6)  # 数值稳定性
    # 混合粘度：铜为固体，mu_Cu取1e10抑制流动
    mu_total = rho_x * mu_Cu + (1.0 - rho_x) * mu_PCM
    # 运动粘度ν = μ/ρ
    nu = mu_total / rho_total

    # -------------------------- 5. 定压比热容cp_total--------------------------
    # D(T)：高斯函数，平滑化相变潜热
    sigma = DeltaT / 4.0  # 标准差
    D_T = torch.exp(-((T - Tpc) ** 2) / (sigma ** 2)) / (torch.sqrt(torch.tensor(np.pi)) * sigma)
    D_T = torch.clamp(D_T, 0.0, 1e3)  # 裁剪D_T
    # PCM比热容
    cp_PCM = cp_s + phi * (cp_l - cp_s) + L * D_T
    # 混合比热容：质量加权平均
    mass_Cu = rho_x * rho_Cu
    mass_PCM = (1.0 - rho_x) * rho_PCM
    mass_total = mass_Cu + mass_PCM + 1e-10
    cp_total = (mass_Cu * cp_Cu + mass_PCM * cp_PCM) / mass_total

    # -------------------------- 6. 热扩散率a--------------------------
    a = lambda_total / (rho_total * cp_total + 1e-10)

    return rho_total, lambda_total, mu_total, nu, cp_total, a, S_t

In [10]:
def sample_collocation_points(N):
    """采样设计域内配点（x,y,τ）+ 面积元权重，用于PDE损失计算"""
    # 1. 时间τ采样：τ∈[0, 1e5s]，归一化到[0,1]便于训练
    tau = np.random.uniform(0.0, 1e5, size=(N, 1))  # [N,1]
    tau_norm = tau / 1e5  # 归一化到[0,1]

    # 2. 环形域（x,y）采样：极坐标转换（避免采样到域外）
    r = np.random.uniform(r0, r1, size=(N, 1))  # 半径∈[r0, r1]
    theta = np.random.uniform(0.0, 2*np.pi, size=(N, 1))  # 角度∈[0,2π]
    x = r * np.cos(theta)  # x坐标
    y = r * np.sin(theta)  # y坐标

    # 3. 输入归一化：x,y∈[-r1, r1]→归一化到[-1,1]，提升训练稳定性
    x_norm = x / r1
    y_norm = y / r1

    # 4. 面积元权重：环形域dA=r*dr*dθ，假设dr/dθ均匀，权重≈r（归一化后）
    weight = r / np.mean(r)  # 归一化权重（总和为N，便于平均）

    # 合并为输入张量：[N, 3] = (x_norm, y_norm, tau_norm)
    points = np.hstack([x_norm, y_norm, tau_norm])
    return (torch.tensor(points, dtype=torch.float32).to(device),
            torch.tensor(weight, dtype=torch.float32).to(device))  # 返回（配点，权重）

def adaptive_sample_simple(N, prev_points, prev_weights, prev_residuals):
    """
    自适应采样：保留高残差区域的部分点
    """
    # 选择前30%的高残差点
    k = int(N * 0.3)
    _, indices = torch.topk(prev_residuals.squeeze(), k)
    
    # 保留高残差点
    high_residual_points = prev_points[indices]
    high_residual_weights = prev_weights[indices]
    
    # 生成新的随机点（占70%）
    new_N = N - k
    new_points, new_weights = sample_collocation_points(new_N)
    
    # 合并
    adaptive_points = torch.cat([new_points, high_residual_points], dim=0)
    adaptive_weights = torch.cat([new_weights, high_residual_weights], dim=0)
    
    return adaptive_points, adaptive_weights

def sample_initial_condition(N):
    """采样初始条件点（τ=0, x,y∈设计域）+ 面积元权重"""
    # 时间τ=0，归一化后为0
    tau_norm = np.zeros((N, 1))

    # 环形域（x,y）采样（同配点采样逻辑）
    r = np.random.uniform(r0, r1, size=(N, 1))
    theta = np.random.uniform(0.0, 2*np.pi, size=(N, 1))
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    x_norm = x / r1
    y_norm = y / r1

    # 面积元权重：环形域dA=r*dr*dθ，假设dr/dθ均匀，权重≈r（归一化后）
    weight = r / np.mean(r)

    points = np.hstack([x_norm, y_norm, tau_norm])
    return (torch.tensor(points, dtype=torch.float32).to(device),
            torch.tensor(weight, dtype=torch.float32).to(device))  # 返回（初始点，权重）

def sample_boundary(N1, N2):
    """采样边界点：N1=内管壁（Dirichlet），N2=外壳（Neumann）"""
    # 1. 内管壁（x²+y²=r0²）
    theta1 = np.random.uniform(0.0, 2*np.pi, size=(N1, 1))
    x1 = r0 * np.cos(theta1)
    y1 = r0 * np.sin(theta1)
    x1_norm = x1 / r1
    y1_norm = y1 / r1
    tau1_norm = np.random.uniform(0.0, 1.0, size=(N1, 1))  # 时间∈[0,1]（归一化后）
    bc1_points = np.hstack([x1_norm, y1_norm, tau1_norm])

    # 2. 外壳（x²+y²=r1²）
    theta2 = np.random.uniform(0.0, 2*np.pi, size=(N2, 1))
    x2 = r1 * np.cos(theta2)
    y2 = r1 * np.sin(theta2)
    x2_norm = x2 / r1
    y2_norm = y2 / r1
    tau2_norm = np.random.uniform(0.0, 1.0, size=(N2, 1))
    bc2_points = np.hstack([x2_norm, y2_norm, tau2_norm])

    return (torch.tensor(bc1_points, dtype=torch.float32).to(device),
            torch.tensor(bc2_points, dtype=torch.float32).to(device))

In [11]:
def compute_volume_constraint_loss(model):
    """计算体积约束损失，仅使用空间采样点"""
    # 仅采样空间点，不包含时间维度
    N_vol = 5000
    r = np.random.uniform(r0, r1, size=(N_vol, 1))
    theta = np.random.uniform(0.0, 2*np.pi, size=(N_vol, 1))
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    x_norm = x / r1
    y_norm = y / r1
    
    # 创建空间点张量（时间维度设为0）
    x_space = torch.tensor(np.hstack([x_norm, y_norm]), dtype=torch.float32).to(device)
    x_with_time = torch.cat([x_space, torch.zeros(N_vol, 1).to(device)], dim=1)
    
    # 获取拓扑变量
    _, _, _, _, rho_x = model(x_with_time)
    
    # 计算体积比（面积加权平均）
    weights = torch.tensor(r / np.mean(r), dtype=torch.float32).to(device)
    rho_x_avg = torch.sum(rho_x * weights) / torch.sum(weights)
    
    # 体积约束损失
    vol_residual = torch.relu(rho_x_avg - phi_total)  # 仅惩罚超过约束的部分
    L_rho_vol = vol_residual ** 2 * 100.0  # 增加权重确保约束满足
    
    return L_rho_vol, rho_x_avg

In [12]:
def compute_loss(model, collocation_points, collocation_weights, ic_points, ic_weights, bc1_points, bc2_points, heat_storage=True, compute_objective=True):
    """
    计算总损失L_total = λ1*L_PDE + λ2*L_IC/BC + λ3*L_topo-constraint + λ4*L_objective
    heat_storage: True=储热过程，False=释热过程
    compute_objective: 是否计算优化目标损失
    """
    # -------------------------- 1. 预处理：获取各类采样点的网络输出 --------------------------
    # 配点输出（用于PDE损失）
    model.train()
    x_col = collocation_points.requires_grad_(True)  # 需计算梯度，开启autograd
    ux_col, uy_col, p_col, T_col, rho_x_col = model(x_col)
    rho_x_col = torch.clamp(rho_x_col, 1e-6, 1.0 - 1e-6)
    T_col = torch.clamp(T_col, 285.0, 365.0)  # 约束温度
    # 计算热物性参数
    rho_total, lambda_total, mu_total, nu_col, cp_total, a_col, S_t = compute_thermo_props(T_col, rho_x_col)

    # 初始条件点输出（用于IC损失）
    x_ic = ic_points.requires_grad_(True)
    ux_ic, uy_ic, p_ic, T_ic, rho_x_ic = model(x_ic)

    # 边界点输出（用于BC损失）
    x_bc1 = bc1_points.requires_grad_(True)  # 内管壁（Dirichlet）
    ux_bc1, uy_bc1, p_bc1, T_bc1, rho_x_bc1 = model(x_bc1)

    x_bc2 = bc2_points.requires_grad_(True)  # 外壳（Neumann）
    ux_bc2, uy_bc2, p_bc2, T_bc2, rho_x_bc2 = model(x_bc2)

    # -------------------------- 2. PDE损失L_PDE--------------------------
    # 2.1 质量守恒损失L_mass
    grad_ux = torch.autograd.grad(ux_col, x_col, grad_outputs=torch.ones_like(ux_col),
                                  create_graph=True, retain_graph=True)[0]
    dudx = grad_ux[:, 0:1]  # ux对x的偏导
    dudy = grad_ux[:, 1:2]  # ux对y的偏导

    grad_uy = torch.autograd.grad(uy_col, x_col, grad_outputs=torch.ones_like(uy_col),
                                  create_graph=True, retain_graph=True)[0]
    dvdy = grad_uy[:, 1:2]  # uy对y的偏导
    dvdx = grad_uy[:, 0:1]  # uy对x的偏导

    mass_residual = dudx + dvdy  # 质量守恒残差：∂ux/∂x + ∂uy/∂y
    L_mass = torch.sum(mass_residual ** 2 * collocation_weights) / torch.sum(collocation_weights)

    # 2.2 动量守恒损失L_momentum
    # 对流项：u·∇ux = ux*∂ux/∂x + uy*∂ux/∂y；u·∇uy = ux*∂uy/∂x + uy*∂uy/∂y
    convect_ux = ux_col * dudx + uy_col * dudy
    convect_uy = ux_col * dvdx + uy_col * dvdy

    # 压力梯度项：-1/ρ * ∂p/∂x；-1/ρ * ∂p/∂y
    grad_p = torch.autograd.grad(p_col, x_col, grad_outputs=torch.ones_like(p_col),
                                 create_graph=True, retain_graph=True)[0]
    dpdx = grad_p[:, 0:1]
    dpdy = grad_p[:, 1:2]
    pressure_term_x = -dpdx / rho_total
    pressure_term_y = -dpdy / rho_total

    # 粘性项：ν∇²ux = ν(∂²ux/∂x² + ∂²ux/∂y²)；ν∇²uy = ν(∂²uy/∂x² + ∂²uy/∂y²)
    d2udx2 = torch.autograd.grad(dudx, x_col, grad_outputs=torch.ones_like(dudx),
                                 create_graph=True, retain_graph=True)[0][:, 0:1]
    d2udy2 = torch.autograd.grad(dudy, x_col, grad_outputs=torch.ones_like(dudy),
                                 create_graph=True, retain_graph=True)[0][:, 1:2]
    viscous_term_x = nu_col * (d2udx2 + d2udy2)

    d2vdx2 = torch.autograd.grad(dvdx, x_col, grad_outputs=torch.ones_like(dvdx),
                                 create_graph=True, retain_graph=True)[0][:, 0:1]
    d2vdy2 = torch.autograd.grad(dvdy, x_col, grad_outputs=torch.ones_like(dvdy),
                                 create_graph=True, retain_graph=True)[0][:, 1:2]
    viscous_term_y = nu_col * (d2vdx2 + d2vdy2)

    # 糊状区源项：-S_t*ux；-S_t*uy
    source_term_x = -S_t * ux_col
    source_term_y = -S_t * uy_col

    # ==================== 浮力项修正（方案B）====================
    # 原代码：F_B = rho_l * alpha * g * (T_col - Tpc) * (1.0 - rho_x_col) * phi_col
    # 修正为：浮力项仅存在于PCM区域（1-ρ_x），但不乘以液相率φ
    # 浮力项（仅y方向）：F_B = ρ_l * α * g * (T - T_pc) * (1-ρ_x)
    phi_col = model.compute_phi_from_T(T_col)
    F_B = rho_l * alpha * g * (T_col - Tpc) * (1.0 - rho_x_col)  # 移除了*phi_col

    # 动量残差（x,y方向）- 方程形式：convect + pressure/rho - viscous - source - F_B = 0
    mom_residual_x = convect_ux + pressure_term_x - viscous_term_x - source_term_x
    mom_residual_y = convect_uy + pressure_term_y - viscous_term_y - source_term_y - F_B
    L_momentum = torch.sum((mom_residual_x ** 2 + mom_residual_y ** 2) * collocation_weights) / torch.sum(collocation_weights)

    # 2.3 传热控制损失L_heat（∂T/∂τ + u·∇T - a∇²T=0）
    grad_T = torch.autograd.grad(T_col, x_col, grad_outputs=torch.ones_like(T_col),
                             create_graph=True, retain_graph=True)[0]
    dTdx = torch.clamp(grad_T[:, 0:1], -100.0, 100.0)  # 梯度裁剪
    dTdy = torch.clamp(grad_T[:, 1:2], -100.0, 100.0)
    dTdtau_norm = grad_T[:, 2:3]  # 对归一化时间（τ_norm=τ/1e5）的导数

    # 时间导数量级转换：∂T/∂τ（实际时间）= ∂T/∂τ_norm * dτ_norm/dτ = dTdtau_norm / 1e5
    dTdtau_real = dTdtau_norm / 1e5

    # 对流项：u·∇T = ux*∂T/∂x + uy*∂T/∂y
    convect_T = ux_col * dTdx + uy_col * dTdy

    # 扩散项：a∇²T = a(∂²T/∂x² + ∂²T/∂y²)
    d2Tdx2 = torch.autograd.grad(dTdx, x_col, grad_outputs=torch.ones_like(dTdx),
                                 create_graph=True, retain_graph=True)[0][:, 0:1]
    d2Tdy2 = torch.autograd.grad(dTdy, x_col, grad_outputs=torch.ones_like(dTdy),
                                 create_graph=True, retain_graph=True)[0][:, 1:2]
    d2Tdx2 = torch.clamp(d2Tdx2, -1e4, 1e4)  # 数值稳定性
    d2Tdy2 = torch.clamp(d2Tdy2, -1e4, 1e4)
    diffusive_term = a_col * (d2Tdx2 + d2Tdy2)

    # 传热残差（使用实际时间导数）
    heat_residual = dTdtau_real + convect_T - diffusive_term
    L_heat = torch.sum(heat_residual ** 2 * collocation_weights) / torch.sum(collocation_weights)

    # 压力正则化：避免压力数值过大
    L_pressure_reg = torch.mean(p_col ** 2) * 1e-6
    # PDE总损失
    L_PDE = L_mass + L_momentum + L_heat + L_pressure_reg

    # -------------------------- 3. 初始/边界条件损失L_IC/BC--------------------------
    # 3.1 初始条件损失L_IC（τ=0）
    T0 = 290.0 if heat_storage else 360.0  # 储热T0=290K，释热T0=360K
    L_IC = torch.sum(((T_ic - T0) ** 2 * 10 + ux_ic ** 2 + uy_ic ** 2) * ic_weights) / torch.sum(ic_weights)

    # 3.2 边界条件损失L_BC
    # 内管壁（Dirichlet）：T=360K（储热）或290K（释热）
    Tw = 360.0 if heat_storage else 290.0
    L_BC1 = torch.mean((T_bc1 - Tw) ** 2 * 2)

    # 外壳（Neumann）：绝热，∂T/∂n=0（法向导数为0）
    grad_T_bc2 = torch.autograd.grad(T_bc2, x_bc2, grad_outputs=torch.ones_like(T_bc2), create_graph=True, retain_graph=True)[0]
    dTdx_bc2 = grad_T_bc2[:, 0:1]
    dTdy_bc2 = grad_T_bc2[:, 1:2]
    # 注意：x_bc2是归一化坐标，需要转换为实际导数
    dTdx_bc2_actual = dTdx_bc2 / r1
    dTdy_bc2_actual = dTdy_bc2 / r1
    
    x_real = x_bc2[:, 0:1] * r1
    y_real = x_bc2[:, 1:2] * r1
    r_norm = torch.sqrt(x_real**2 + y_real**2 + 1e-10)
    
    # 法向导数：∇T·n = (∂T/∂x)*(x/r) + (∂T/∂y)*(y/r)
    dTdn = (x_real/r_norm) * dTdx_bc2_actual + (y_real/r_norm) * dTdy_bc2_actual
    L_BC2 = torch.mean(dTdn**2)

    # IC/BC总损失
    L_IC_BC = L_IC + L_BC1 + L_BC2

    # -------------------------- 4. 拓扑约束损失L_topo-constraint--------------------------
    # 4.1 ρx取值约束（ρx∈[0,1]）
    L_rho_bounds = torch.sum((torch.max(torch.tensor(0.0).to(device), -rho_x_col) ** 2 +
                          torch.max(torch.tensor(0.0).to(device), rho_x_col - 1.0) ** 2) * collocation_weights) / torch.sum(collocation_weights)

    # 4.2 体积比约束（单独计算）
    L_rho_vol, rho_x_avg = compute_volume_constraint_loss(model)

    # 拓扑约束总损失
    L_topo_constraint = L_rho_bounds + L_rho_vol

    # -------------------------- 5. 优化目标损失L_objective（方案D修正）--------------------------
    L_objective = torch.tensor(0.0).to(device)
    if compute_objective:
        # 采样优化目标点（设计域内）+ 面积元权重，开启自动求导
        x_obj, weight_obj = sample_collocation_points(N_obj)
        x_obj = x_obj.requires_grad_(True)
        ux_obj, uy_obj, p_obj, T_obj, rho_x_obj = model(x_obj)
        T_obj = torch.clamp(T_obj, 285.0, 365.0)  # 约束温度
        weight_obj = weight_obj.unsqueeze(1)  # [N_obj,1]，匹配张量维度

        # Case1：最小平均温度（加权平均）
        T_ave = torch.sum(T_obj * weight_obj) / torch.sum(weight_obj)
        L_obj1 = T_ave ** 2

        # Case2：最小温度均方差（加权均方差）
        T_var = torch.sum(((T_obj - T_ave) ** 2) * weight_obj) / torch.sum(weight_obj)
        L_obj2 = T_var

        # Case3：多目标（平均温度+温度均方差+火积耗散）
        # ==================== 火积耗散修正（方案D）====================
        # 原代码：φ_g = ∫λ|∇T|² dA（错误）
        # 修正为：φ_g = ∫λ∇²T dA（按论文式(16)）
        # 计算拉普拉斯算子：∇²T = ∂²T/∂x² + ∂²T/∂y²
        grad_T_obj = torch.autograd.grad(T_obj, x_obj, grad_outputs=torch.ones_like(T_obj),
                                         create_graph=True, retain_graph=True)[0]
        dTdx_obj = grad_T_obj[:, 0:1]
        dTdy_obj = grad_T_obj[:, 1:2]
        
        # 计算二阶导数
        d2Tdx2_obj = torch.autograd.grad(dTdx_obj, x_obj, grad_outputs=torch.ones_like(dTdx_obj),
                                         create_graph=True, retain_graph=True)[0][:, 0:1]
        d2Tdy2_obj = torch.autograd.grad(dTdy_obj, x_obj, grad_outputs=torch.ones_like(dTdy_obj),
                                         create_graph=True, retain_graph=True)[0][:, 1:2]
        
        # 拉普拉斯项
        laplacian_T = d2Tdx2_obj + d2Tdy2_obj
        
        # 计算混合导热系数
        _, lambda_total_obj, _, _, _, _, _ = compute_thermo_props(T_obj, rho_x_obj)
        
        # 火积耗散：φ_g = ∫λ∇²T dA（按论文式(16)）
        phi_g = torch.sum(lambda_total_obj * laplacian_T * weight_obj) / torch.sum(weight_obj)
        L_obj3 = w1 * L_obj1 + w2 * L_obj2 + w3 * phi_g ** 2

        # 选择对应Case的目标损失
        if case == 1:
            L_objective = L_obj1
        elif case == 2:
            L_objective = L_obj2
        else:
            L_objective = L_obj3

    # -------------------------- 6. 总损失（方案A权重调整） --------------------------
    # 使用调整后的权重：λ1=1.0, λ2=1.0, λ3=100.0, λ4=0.1
    if compute_objective:
        L_total = lambda1 * L_PDE + lambda2 * L_IC_BC + lambda3 * L_topo_constraint + lambda4 * L_objective
    else:
        L_total = lambda1 * L_PDE + lambda2 * L_IC_BC + lambda3 * L_topo_constraint

    # 返回各损失项（用于训练监控）
    return L_total, L_PDE, L_IC_BC, L_topo_constraint, L_objective

In [13]:
def train_model(model, epochs_pre=5000, epochs_fine=1000):
    """训练流程：分阶段训练"""
    # 第一阶段：仅满足物理约束
    print("="*50)
    print("第一阶段：满足基本物理约束")
    print("="*50)
    
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=100)
    
    # 初始化采样点
    collocation_points, collocation_weights = sample_collocation_points(N_mass)
    ic_points, ic_weights = sample_initial_condition(N_IC)
    bc1_points, bc2_points = sample_boundary(N_BC1, N_BC2)
    
    # 第一阶段训练（不计算优化目标）
    for epoch in range(epochs_pre):
        # 每200轮重新采样
        if epoch % 200 == 0 and epoch != 0:
            collocation_points, collocation_weights = sample_collocation_points(N_mass)
            ic_points, ic_weights = sample_initial_condition(N_IC)
            bc1_points, bc2_points = sample_boundary(N_BC1, N_BC2)
        
        # 计算损失（屏蔽优化目标）
        L_total, L_PDE, L_IC_BC, L_topo, _ = compute_loss(
            model, collocation_points, collocation_weights, 
            ic_points, ic_weights, bc1_points, bc2_points, 
            heat_storage=True, compute_objective=False
        )
        
        # 反向传播
        optimizer.zero_grad()
        L_total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step(L_total)
        
        # 打印损失
        if (epoch + 1) % 100 == 0:
            _, rho_x_avg = compute_volume_constraint_loss(model)
            print(f"预训练Epoch [{epoch+1}/{epochs_pre}] | "
                  f"总损失: {L_total.item():.4f} | "
                  f"PDE损失: {L_PDE.item():.4f} | "
                  f"IC/BC损失: {L_IC_BC.item():.4f} | "
                  f"拓扑约束损失: {L_topo.item():.4f} | "
                  f"体积比: {rho_x_avg.item():.3f}")
    
    # 第二阶段：引入优化目标
    print("\n" + "="*50)
    print("第二阶段：优化目标训练")
    print("="*50)
    
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    
    # 第二阶段训练（计算优化目标）
    for epoch in range(epochs_fine):
        # 动态采样（每50轮）
        if epoch % 50 == 0:
            # 计算传热残差用于自适应采样
            x_col_temp = collocation_points.requires_grad_(True)
            ux_col_temp, uy_col_temp, p_col_temp, T_col_temp, rho_x_col_temp = model(x_col_temp)
            rho_total_temp, lambda_total_temp, mu_total_temp, nu_col_temp, cp_total_temp, a_col_temp, S_t_temp = compute_thermo_props(T_col_temp, rho_x_col_temp)
            
            # 计算传热残差
            grad_T_temp = torch.autograd.grad(T_col_temp, x_col_temp, grad_outputs=torch.ones_like(T_col_temp), create_graph=True, retain_graph=True)[0]
            dTdx_temp = grad_T_temp[:, 0:1]
            dTdy_temp = grad_T_temp[:, 1:2]
            dTdtau_norm_temp = grad_T_temp[:, 2:3]
            dTdtau_real_temp = dTdtau_norm_temp / 1e5
            convect_T_temp = ux_col_temp * dTdx_temp + uy_col_temp * dTdy_temp
            
            d2Tdx2_temp = torch.autograd.grad(dTdx_temp, x_col_temp, grad_outputs=torch.ones_like(dTdx_temp), create_graph=False, retain_graph=True)[0][:, 0:1]
            d2Tdy2_temp = torch.autograd.grad(dTdy_temp, x_col_temp, grad_outputs=torch.ones_like(dTdy_temp), create_graph=False, retain_graph=False)[0][:, 1:2]
            diffusive_term_temp = a_col_temp * (d2Tdx2_temp + d2Tdy2_temp)
            heat_residual_temp = dTdtau_real_temp + convect_T_temp - diffusive_term_temp
            
            # 自适应采样更新配点和权重
            collocation_points, collocation_weights = adaptive_sample_simple(N_mass, collocation_points, collocation_weights, heat_residual_temp)
            bc1_points, bc2_points = sample_boundary(N_BC1, N_BC2)
            ic_points, ic_weights = sample_initial_condition(N_IC)
        
        # 计算损失（包含优化目标）
        L_total, L_PDE, L_IC_BC, L_topo, L_obj = compute_loss(
            model, collocation_points, collocation_weights,
            ic_points, ic_weights, bc1_points, bc2_points,
            heat_storage=True, compute_objective=True
        )
        
        # 反向传播
        optimizer.zero_grad()
        L_total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()
        
        # 打印损失
        if (epoch + 1) % 50 == 0:
            _, rho_x_avg = compute_volume_constraint_loss(model)
            print(f"优化训练Epoch [{epoch+1}/{epochs_fine}] | "
                  f"总损失: {L_total.item():.4f} | "
                  f"PDE损失: {L_PDE.item():.4f} | "
                  f"目标损失: {L_obj.item():.4f} | "
                  f"体积比: {rho_x_avg.item():.3f}")
    
    # 保存训练好的模型
    torch.save(model.state_dict(), f"topo_pinn_case{case}.pth")
    print("\n训练完成！模型已保存为 topo_pinn_case{case}.pth")

In [14]:
def compute_key_metrics(model, heat_storage=True):
    """计算关键指标，与论文结果对比"""
    model.eval()
    
    # 选择关键时间点
    tau_list = [100, 300, 500, 751, 1000]
    metrics = {}
    
    for tau in tau_list:
        tau_norm = tau / 1e5
        
        # 采样点计算平均温度
        x_obj, weight_obj = sample_collocation_points(5000)
        x_obj = x_obj.clone()
        x_obj[:, 2] = tau_norm  # 设置固定时间
        
        with torch.no_grad():
            ux, uy, p, T, rho_x = model(x_obj)
            phi = model.compute_phi_from_T(T)
            
            # 平均温度
            T_avg = torch.sum(T * weight_obj) / torch.sum(weight_obj)
            
            # 平均液相率
            phi_avg = torch.sum(phi * weight_obj) / torch.sum(weight_obj)
            
            metrics[f"T_avg_{tau}s"] = T_avg.item()
            metrics[f"phi_avg_{tau}s"] = phi_avg.item()
    
    return metrics

In [15]:
def visualize_results(model, save_path="./results"):
    """可视化结果：温度云图、液相率云图、拓扑结构（rho_x）"""
    import os
    os.makedirs(save_path, exist_ok=True)
    
    # 生成环形域网格点
    r = np.linspace(r0, r1, 100)
    theta = np.linspace(0, 2*np.pi, 100)
    R, Theta = np.meshgrid(r, theta)
    X = R * np.cos(Theta)
    Y = R * np.sin(Theta)
    
    # 选择关键时间点可视化（储热过程）
    tau_list = [100, 300, 500, 751, 1000]
    for tau in tau_list:
        tau_norm = tau / 1e5
        # 构造输入（x_norm, y_norm, tau_norm）
        x_input = np.hstack([
            X.reshape(-1, 1)/r1,
            Y.reshape(-1, 1)/r1,
            np.full((10000, 1), tau_norm)
        ])
        x_tensor = torch.tensor(x_input, dtype=torch.float32).to(device)
        
        with torch.no_grad():
            ux, uy, p, T, rho_x = model(x_tensor)
            phi = model.compute_phi_from_T(T)
        
        # 转换为网格格式
        T_grid = T.detach().cpu().numpy().reshape(100, 100)
        phi_grid = phi.detach().cpu().numpy().reshape(100, 100)
        rho_x_grid = rho_x.detach().cpu().numpy().reshape(100, 100)
        
        # 绘制温度云图
        plt.figure(figsize=(12, 4))
        plt.subplot(1, 3, 1)
        contourf = plt.contourf(X, Y, T_grid, cmap='jet', vmin=290, vmax=360)
        plt.colorbar(contourf, label='Temperature (K)')
        plt.title(f'Temperature (τ={tau}s)')
        plt.axis('equal')
        
        # 绘制液相率云图
        plt.subplot(1, 3, 2)
        contourf = plt.contourf(X, Y, phi_grid, cmap='viridis', vmin=0, vmax=1)
        plt.colorbar(contourf, label='Liquid Fraction φ')
        plt.title(f'Liquid Fraction (τ={tau}s)')
        plt.axis('equal')
        
        # 绘制拓扑结构（rho_x≥0.5为铜，否则为PCM）
        plt.subplot(1, 3, 3)
        contourf = plt.contourf(X, Y, (rho_x_grid >= 0.5).astype(int), cmap='binary')
        plt.colorbar(contourf, label='Topology (1=Cu, 0=PCM)')
        plt.title(f'Topological Structure (τ={tau}s)')
        plt.axis('equal')
        
        plt.tight_layout()
        plt.savefig(os.path.join(save_path, f'result_tau_{tau}s.png'), dpi=300, bbox_inches='tight')
        plt.close()
    
    print(f"可视化结果已保存到 {save_path}")

In [16]:
if __name__ == "__main__":
    # 初始化模型
    model = TopoPINN(hidden_layers=6, hidden_dim=256).to(device)
    
    # 启动训练（预训练5000轮，精细优化1000轮）
    train_model(model, epochs_pre=5000, epochs_fine=1000)
    
    # 加载训练好的模型
    model.load_state_dict(torch.load(f"topo_pinn_case{case}.pth"))
    model.eval()  # 切换为评估模式
    
    # 计算关键指标，与论文对比
    print("\n" + "="*50)
    print("关键指标验证（对标论文Case3）")
    print("="*50)
    metrics = compute_key_metrics(model, heat_storage=True)
    for key, value in metrics.items():
        if value is not None:
            print(f"{key}: {value:.2f}")
        else:
            print(f"{key}: 未计算成功")
    
    # 可视化结果
    visualize_results(model, save_path="./topo_pinn_results")
    print("\n训练+验证+可视化完成！")

第一阶段：满足基本物理约束
预训练Epoch [100/5000] | 总损失: 8700530.0000 | PDE损失: 8690541.0000 | IC/BC损失: 9989.2842 | 拓扑约束损失: 0.0000 | 体积比: 0.000
预训练Epoch [200/5000] | 总损失: 14573940.0000 | PDE损失: 14564726.0000 | IC/BC损失: 9214.4785 | 拓扑约束损失: 0.0000 | 体积比: 0.000
预训练Epoch [300/5000] | 总损失: 621148.6875 | PDE损失: 613189.8125 | IC/BC损失: 7958.8584 | 拓扑约束损失: 0.0000 | 体积比: 0.000
预训练Epoch [400/5000] | 总损失: 3484988.2500 | PDE损失: 3477418.0000 | IC/BC损失: 7570.2769 | 拓扑约束损失: 0.0000 | 体积比: 0.000
预训练Epoch [500/5000] | 总损失: 1230882.8750 | PDE损失: 1210887.0000 | IC/BC损失: 19995.9023 | 拓扑约束损失: 0.0000 | 体积比: 0.000


KeyboardInterrupt: 